Voice to text


In [ ]:
!sudo apt-get update && sudo apt-get install ffmpeg -y

Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:4 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [89.0 kB]
Get:5 https://cli.github.com/packages stable/main amd64 Packages [356 B]
Get:6 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,608 kB]
Get:7 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:8 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:9 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:10 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:11 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [2,990 kB]
Get:12 http://security.ubuntu.com/ubuntu jammy-security/universe amd64 Packages [1,292 kB]
Get:13 http://archive.ubuntu.com/ubuntu jammy-

In [ ]:
!pip install git+https://github.com/openai/whisper.git

In [ ]:
import whisper

model = whisper.load_model("medium")

rezultat = model.transcribe("/content/A1-0002_Audio1_00086952.wav")

print("\n Text: ")
print(rezultat["text"])

In [ ]:
!pip install evaluate jiwer

import evaluate

# Încărcăm metrica Word Error Rate
wer_metric = evaluate.load("wer")

# Exemplu de evaluare:
text_real = ["salut mă bucur să te cunosc"]
text_generat_de_whisper = ["salut mă bucur să te recunosc"]

# Calculăm scorul (Înmulțim cu 100 ca să îl vedem în procente)
scor_wer = wer_metric.compute(predictions=text_generat_de_whisper, references=text_real) * 100

print(f"Scorul WER este: {scor_wer:.2f}% (Cât mai mic = Cât mai bine)")

Text cleaning

In [ ]:
import re

def curata_text_whisper(text):
    if not text or not isinstance(text, str):
        return ""

    # 1. Eliminăm tag-urile de acțiuni/zgomote generate de Whisper
    # Ex: [Muzică], (tușește), *râde*
    text = re.sub(r'\[.*?\]|\(.*?\)|\*.*?\*', '', text)

    # 2. Eliminăm ezitările și cuvintele de umplutură (adaptat RO/EN)
    # Acoperă variații de: ăă, ăăă, îî, uh, um, hm
    fillers = r'\b(ăă+|îî+|uhm+|uh+|um+|hm+)\b'
    text = re.sub(fillers, '', text, flags=re.IGNORECASE)

    # 3. Corectăm spațiile dinaintea semnelor de punctuație
    # Ex: "salut , ce faci ?" -> "salut, ce faci?"
    text = re.sub(r'\s+([.,?!:;])', r'\1', text)

    # 4. Eliminăm repetițiile consecutive de cuvinte (halucinații scurte Whisper)
    # Ex: "Eu am am mers acolo" -> "Eu am mers acolo"
    text = re.sub(r'\b(\w+)(?:\s+\1\b)+', r'\1', text, flags=re.IGNORECASE)

    # 5. Reducem spațiile multiple la un singur spațiu și curățăm capetele
    text = re.sub(r'\s+', ' ', text).strip()

    # 6. (Opțional) Capitalizăm prima literă dacă s-a pierdut din cauza curățării
    if text:
        text = text[0].upper() + text[1:]

    return text

Text summarization


In [ ]:
!pip install --upgrade torch accelerate

In [ ]:
import json
path_test = '/content/test_data.json'
path_train = '/content/train_data.json'

In [ ]:
with open(path_test, 'r', encoding='utf-8') as f:
    data_test = json.load(f)

print(data_test[0])

with open(path_train, 'r', encoding='utf-8') as f:
    data_train = json.load(f)

print(data_train[0])

{'text_complicat': 'Prin urmare, activitatea de soluţionare a contestaţiilor are asigurată autonomia necesară în raport de organele de control fiscal în aşa fel încât să fie posibilă luarea unor decizii legale şi temeinice care să răspundă la toate solicitările contribuabililor.', 'text_simplu': 'Departamentul care rezolvă aceste contestații este independent de inspectorii care fac controalele, tocmai pentru a garanta decizii corecte și obiective.'}
{'text_complicat': 'În acest articol arătăm că o mare parte din revistele de categorie B nu îşi îndeplinesc funcţia de diseminare a informaţiei ştiinţifice nici măcar la nivel naţional, neputând fi găsite în bibliotecile centrale universitare (BCU) sau pe internet.', 'text_simplu': 'Acest articol arată că multe reviste din categoria B nu reușesc să răspândească informația în țară, deoarece nu pot fi găsite nici în biblioteci, nici pe internet.'}


In [ ]:
for i in data_test:
  print(i['text_simplu'])

In [ ]:

import pandas as pd

df = pd.read_json('test_data.json')
print(df.head())

                                      text_complicat  \
0  Prin urmare, activitatea de soluţionare a cont...   
1  Soluţionarea contestaţiilor se face pe baza co...   
2  Pe parcursul instrumentării cauzei, pentru a s...   
3  Procedura de soluţionare a contestaţiilor stab...   
4  Mai mult decât atât, în timp ce în legislaţia ...   

                                         text_simplu  
0  Departamentul care rezolvă aceste contestații ...  
1  Contestațiile se judecă strict pe baza actului...  
2  Pentru a lămuri complet o situație, cei care r...  
3  Regulile românești pentru rezolvarea contestaț...  
4  Pentru a nu lungi procesul, legea noastră ofer...  


In [ ]:
data_test


In [ ]:
from datasets import Dataset
hf_data_test = Dataset.from_pandas(pd.DataFrame(data_test))
hf_data_train = Dataset.from_pandas(pd.DataFrame(data_train))

In [ ]:
!pip install datasets

Model Install

In [ ]:
!pip uninstall torch torchvision torchaudio -y
!pip install torch torchvision torchaudio

In [ ]:
!pip install transformers==4.43.3 bitsandbytes==0.44.1 peft==0.11.1 accelerate==0.32.1 triton==2.3.1 --force-reinstall

In [ ]:
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training

The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

In [ ]:
!pip install torch

In [ ]:
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121 --upgrade --force-reinstall

In [ ]:
import torch
import transformers
import bitsandbytes

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

tokenizer = AutoTokenizer.from_pretrained("facebook/mbart-large-50-many-to-many-mmt")
model = AutoModelForSeq2SeqLM.from_pretrained("facebook/mbart-large-50-many-to-many-mmt")

tokenizer_config.json:   0%|          | 0.00/529 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/649 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.44G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/261 [00:00<?, ?B/s]

In [ ]:

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, BitsAndBytesConfig
from peft import prepare_model_for_kbit_training, LoraConfig, get_peft_model, TaskType

print(f"Versiune Transformers: {transformers.__version__}")
print(f"Versiune BitsAndBytes: {bitsandbytes.__version__}")

# Am revenit la many-to-many, ideal pentru Română -> Română
nume_model = "facebook/mbart-large-50-many-to-many-mmt"
tokenizer = AutoTokenizer.from_pretrained(nume_model)


# Configurarea 4-bit (Stabilă și optimizată)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16, # <-- Vital pentru a funcționa perfect cu fp16=True
    bnb_4bit_quant_type="nf4"             # Formatul ideal pentru LoRA
)

model = AutoModelForSeq2SeqLM.from_pretrained(
    nume_model,
    quantization_config=bnb_config,
    device_map="auto",
)

model.config.decoder_start_token_id = tokenizer.lang_code_to_id["ro_RO"]

# ==========================================================
# Oprim checkpointing-ul ca să nu ne strice viteza și matematica!
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=False)

# SOLUȚIA OFICIALĂ: Îi spunem modelului să mențină firul matematic activ
# (Înlocuiește complet acel cârlig manual care dădea eroare)
model.enable_input_require_grads()
# ==========================================================

# Atașăm adaptorul LoRA
configuratie_lora = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj", "k_proj", "out_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.SEQ_2_SEQ_LM
)
model = get_peft_model(model, configuratie_lora)

model.print_trainable_parameters()
model.config.use_cache = False

Versiune Transformers: 4.43.3
Versiune BitsAndBytes: 0.44.1
trainable params: 4,718,592 || all params: 615,598,080 || trainable%: 0.7665


In [ ]:
from datasets import load_dataset

print("Se descarcă dataset-ul CNN/DailyMail...")
# Încărcăm versiunea standard 3.0.0
dataset_complet = load_dataset("cnn_dailymail", "3.0.0")

# =========================================================
# ⚠️ TĂIEM DATASET-UL CA SĂ NU NE BLOCHĂM SĂPTĂMÂNI ÎNTREGI
# Luăm 3.000 de texte pentru antrenament și 300 pentru evaluare
# =========================================================
train_dataset = dataset_complet["train"].select(range(3000))
eval_dataset = dataset_complet["validation"].select(range(300))

print(f"Am selectat {len(train_dataset)} texte pentru antrenament.")

# =========================================================
# SCHIMBĂM LIMBA ÎN ENGLEZĂ PENTRU ACEST EXPERIMENT
# =========================================================
tokenizer.src_lang = "en_XX"
tokenizer.tgt_lang = "en_XX"
model.config.decoder_start_token_id = tokenizer.lang_code_to_id["en_XX"]

# =========================================================
# FUNCȚIA DE TRANSFORMARE A TEXTELOR ÎN NUMERE (TOKENIZARE)
# =========================================================
def proceseaza_datele(exemple):
    # 1. Tokenizăm articolele (textul lung pe care îl citește modelul)
    inputs = tokenizer(
        exemple["article"],
        max_length=1024,
        truncation=True,
        padding="max_length"
    )

    # 2. Tokenizăm rezumatele corecte (highlights) pe care trebuie să le învețe
    # Folosim text_target pentru a-i spune că acestea sunt etichetele (labels)
    labels = tokenizer(
        text_target=exemple["highlights"],
        max_length=250,
        truncation=True,
        padding="max_length"
    )

    # Atașăm rezumatele la datele de intrare sub numele de "labels"
    inputs["labels"] = labels["input_ids"]
    return inputs

print("Se procesează textele (tokenizare)...")
# Aplicăm funcția pe dataset-urile noastre decupate
train_tokenizat = train_dataset.map(proceseaza_datele, batched=True, remove_columns=["article", "highlights", "id"])
eval_tokenizat = eval_dataset.map(proceseaza_datele, batched=True, remove_columns=["article", "highlights", "id"])

print("Datele sunt pregătite perfect pentru antrenament!")

Se descarcă dataset-ul CNN/DailyMail...
Am selectat 3000 texte pentru antrenament.
Se procesează textele (tokenizare)...


Map:   0%|          | 0/3000 [00:00<?, ? examples/s]

Map:   0%|          | 0/300 [00:00<?, ? examples/s]

Datele sunt pregătite perfect pentru antrenament!


In [ ]:
def preprocess_function(batch):
  source = batch["text_complicat"]
  target = batch["text_simplu"]
  source_ids = tokenizer(source, truncation = True, padding = 'max_length', max_length = 500)
  target_ids = tokenizer(target, truncation = True, padding = 'max_length', max_length = 500)

  labels = target_ids['input_ids']
  labels = [[(label if label != tokenizer.pad_token_id else -100) for label in labels_example] for labels_example in labels]

  return{
    "input_ids":source_ids["input_ids"],
    "attention_mask":source_ids["attention_mask"],
    "labels":labels
}


Training

In [ ]:
df_source_train = hf_data_train.map(
    preprocess_function,
    batched=True,
    remove_columns=["text_complicat", "text_simplu"]
)
df_source_test = hf_data_test.map(
    preprocess_function,
    batched=True,
    remove_columns=["text_complicat", "text_simplu"]
)

Map:   0%|          | 0/327 [00:00<?, ? examples/s]

Map:   0%|          | 0/43 [00:00<?, ? examples/s]

In [ ]:
from transformers import TrainingArguments, Trainer, EarlyStoppingCallback
from transformers import DataCollatorForSeq2Seq

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

training_args = TrainingArguments(
    output_dir="/content/mbart-rezumate-ro",
    num_train_epochs=30,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=8,
    fp16=True,
    gradient_checkpointing=True,
    torch_compile=False,
    learning_rate=1e-4,
    optim="paged_adamw_8bit",
    report_to="none",
    weight_decay=0.01,
    warmup_ratio=0.1,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=5,
    load_best_model_at_end=True,
    metric_for_best_model="loss",
    save_total_limit=2
)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=df_source_train,
    eval_dataset=df_source_test,
    data_collator=data_collator,

    callbacks=[EarlyStoppingCallback(early_stopping_patience=5)]
)

/usr/local/lib/python3.12/dist-packages/accelerate/accelerator.py:482: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)


In [ ]:
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:632: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Epoch,Training Loss,Validation Loss
1,2.996200,2.621124
2,2.512700,2.632715
3,1.997400,2.802240


Some non-default generation parameters are set in the model config. These should go into a GenerationConfig file (https://huggingface.co/docs/transformers/generation_strategies#save-a-custom-decoding-strategy-with-your-model) instead. This warning will be raised to an exception in v4.41.
Non-default generation parameters: {'max_length': 200, 'early_stopping': True, 'num_beams': 5, 'forced_eos_token_id': 2}
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:632: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
Some non-default generation parameters are set in the model config. These should go into a GenerationConfig file (htt

In [ ]:
model.save_pretrained('/content/model_directory')
tokenizer.save_pretrained('/content/model_directory')

NameError: name 'model' is not defined

In [ ]:
from peft import PeftModel

In [ ]:
# 1. Definim căile
nume_model_baza = "facebook/mbart-large-50-many-to-many-mmt"
cale_checkpoint = "/content/mbart-rezumate-ro/checkpoint-38" # IMPORTANT: Pune calea reală către folderul tău!

# 2. Încărcăm Tokenizer-ul original
tokenizer = AutoTokenizer.from_pretrained(nume_model_baza)
tokenizer.src_lang = "ro_RO"

# 3. Încărcăm Modelul de Bază (în 4-bit, ca să încapă perfect pe placa video)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

print("Se încarcă modelul de bază...")
model_baza = AutoModelForSeq2SeqLM.from_pretrained(
    nume_model_baza,
    quantization_config=bnb_config,
    device_map={"": 0}
)
model_baza.config.decoder_start_token_id = tokenizer.lang_code_to_id["ro_RO"]

# 4. ATAȘĂM ADAPTORUL LORA (Aici se întâmplă magia!)
print("Se atașează cunoștințele învățate de tine...")
model = PeftModel.from_pretrained(model_baza, cale_checkpoint)

print("Modelul a fost încărcat cu succes și este gata de treabă!")

Se încarcă modelul de bază...
Se atașează cunoștințele învățate de tine...
Modelul a fost încărcat cu succes și este gata de treabă!


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# ========================================================
# 1. SETEAZĂ NUMELE MODELULUI TĂU AICI
# ========================================================
nume_model_hf = "FlorinCatalin1/mBart-50_summary_epoch6" # Ex: "ionut/mbart-rezumate-ro-v1"

print(f"Se descarcă Tokenizer-ul din {nume_model_hf}...")
tokenizer = AutoTokenizer.from_pretrained(nume_model_hf)
# Setăm limba pe română (dacă pentru asta a fost antrenat)
tokenizer.src_lang = "ro_RO"

print("Se descarcă Modelul (aceasta va dura puțin la prima rulare)...")
model = AutoModelForSeq2SeqLM.from_pretrained(
    nume_model_hf,
    device_map="auto",            # Împarte automat modelul pe GPU
    torch_dtype=torch.float16     # Salvează memorie VRAM la inferență
)

# Ne asigurăm că știe cu ce limbă să înceapă decodarea
model.config.decoder_start_token_id = tokenizer.lang_code_to_id["ro_RO"]
print("Model încărcat cu succes!")

Se descarcă Tokenizer-ul din FlorinCatalin1/mBart-50_summary_epoch6...


config.json: 0.00B [00:00, ?B/s]

Se descarcă Modelul (aceasta va dura puțin la prima rulare)...


`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/2.44G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/516 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/255 [00:00<?, ?B/s]

Model încărcat cu succes!


In [ ]:
model = "/content/mbart-rezumate-ro/checkpoint-41/model.safetensors"

In [ ]:
def summarize(blog_post):
    inputs = tokenizer(blog_post, max_length=1024, truncation=True, return_tensors='pt').to("cuda")

    with torch.no_grad(): # Oprește calculele de antrenament și salvează VRAM
        summary_ids = model.generate(
            input_ids=inputs['input_ids'],
            attention_mask=inputs['attention_mask'],
            max_length=250,

            # Armele anti-repetiție
            repetition_penalty=2.5,
            no_repeat_ngram_size=3,

            # Modul logic și factual (Fără do_sample sau temperature)
            num_beams=4,
            early_stopping=True,

            # Reamintirea limbii
            forced_bos_token_id=tokenizer.lang_code_to_id["ro_RO"]
        )

    # Decodăm tokenii înapoi în text, ignorând tokenii speciali
    summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)

    return summary


Rezultate


In [ ]:
blog_post = """
În interiorul stației Chamartín, probabil a doua ca mărime după Atocha, printre fluxuri de navetiști și panouri electronice, se deschide un muzeu care nu rupe vizitatorul de infrastructură, ci îl afundă în ea. Expoziția permanentă urmărește nașterea și evoluția materialului rulant al metroului madrilen, de la primele trenuri introduse în 1919 până la seriile puse în circulație în anii ’60.

Nu este un muzeu al obiectelor moarte, ci unul al mecanismelor readuse la viață. Douăsprezece trenuri istorice, restaurate, formează coloana vertebrală a parcursului. Sunt gravitate de aproape o sută de piese originale: elemente de semnalizare, lămpi, scaune, plăcuțe, aparate, fotografii de epocă. Toate construiesc un fir continuu între metroul interbelic și sistemul automatizat de astăzi.

Din toamna lui 2025, muzeul s-a extins simbolic într-o zonă de suveniruri oficiale: reproduceri ale celebrului romb roșu al intrărilor, obiecte aniversare, articole inspirate din identitatea grafică istorică a companiei. Un gest aparent comercial, dar cu miză culturală: metroul nu este doar funcțional, ci și asumabil ca patrimoniu."""
summary = summarize(blog_post)
print(f'summary:  {summary}')

summary:  


In [ ]:
blog_post2 = """Templul a fost descoperit de arheologi care lucrau la situl Tell el-Farama din orașul antic Pelusium, aflat la marginea estică a Deltei Nilului, în guvernoratul Sinaiul de Nord, au declarat reprezentanții Ministerul Turismului și Antichităților din Egipt, într-un comunicat citat de Live Science.

Datorită poziției sale strategice aproape de gurile Nilului, orașul-port Pelusium a fost folosit ca fortăreață în perioada faraonică și ulterior ca punct vamal în timpul ascensiunii Imperiului Roman. În 2022, arheologii au descoperit la acest sit un templu dedicat lui Zeus, realizat din granit roz.

Experții au descoperit inițial templul dedicat lui Pelusius în 2019. O excavare parțială a aproximativ un sfert din zonă a scos la iveală o structură circulară din cărămidă roșie, pe care arheologii au interpretat-o ca fiind casa senatului orașului, a declarat Hesham Hussein, șeful Administrației Centrale pentru Antichități din Egiptul de Jos și Sinai, în comunicat. Însă, după finalizarea unei excavări complete a templului și expunerea integrală a clădirii, această interpretare s-a schimbat."""
summary2 = summarize(blog_post2)
print(f'summary: {summary2}')

In [ ]:
blog_post3 = """ Domnul
Myriel era fiul unui consilier al curţii de justiţie din Aix; viţă de magistraţi. Se
povesteşte că tatăl lui, hărăzindu-i moştenirea postului său, îl însurase foarte de
tânăr, la optsprezece sau douăzeci de ani, după un obicei destul de răspândit în
familiile magistraţilor. În pofida acestei căsătorii, Charles Myriel făcuse, zice-se,
să se vorbească multe pe seama sa. Era bine făcut, deşi cam mic de statură,
elegant, distins, spiritual; tinereţea lui fusese închinată în întregime vieţii
mondene şi aventurilor galante."""
summary3 = summarize(blog_post3)
print(f'summary : {summary3}')

In [ ]:
!pip install evaluate sacrebleu


In [ ]:
!pip install rouge_score

In [ ]:
import evaluate

metrica_sacrebleu = evaluate.load("sacrebleu")
metrica_rouge = evaluate.load('rouge')
ter_metric = evaluate.load("ter")

rezultat_bleu = metrica_sacrebleu.compute(predictions=[summary2], references=[[blog_post]])
print("Rezultat BLEU:", rezultat_bleu)

rezultat_rouge = metrica_rouge.compute(predictions=[summary2], references=[[blog_post]])
print("Rezultat Rouge:", rezultat_rouge)

In [ ]:
import matplotlib.pyplot as plt

def plot_bleu_score(sacrebleu_dict):

    bleu_score = round(sacrebleu_dict['score'], 2)
    precisions = [round(p, 2) for p in sacrebleu_dict['precisions']]
    bp = round(sacrebleu_dict['bp'], 3)
    sys_len = sacrebleu_dict['sys_len']
    ref_len = sacrebleu_dict['ref_len']


    fig, ax = plt.subplots(figsize=(8, 5))


    x_labels = ['1-gram\n(Cuvinte)', '2-gram\n(Bigrame)', '3-gram\n(Trigrame)', '4-gram\n(Tetragrame)']
    culori = ['#4C72B0', '#55A868', '#C44E52', '#8172B2']
    bars = ax.bar(x_labels, precisions, color=culori)


    for bar in bars:
        yval = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2, yval + 1.5, f'{yval}%',
                ha='center', va='bottom', fontweight='bold')


    ax.set_ylim(0, 100)
    ax.set_ylabel('Precizie (%)', fontsize=12)
    ax.set_title('Analiza Preciziilor N-grame (sacreBLEU)', fontsize=14, fontweight='bold')
    ax.grid(axis='y', linestyle='--', alpha=0.7)


    summary_text = (f"Scor BLEU Final: {bleu_score}\n"
                    f"Penalizare Brevitate (BP): {bp}\n"
                    f"Cuvinte generate: {sys_len}\n"
                    f"Cuvinte referință: {ref_len}")

    props = dict(boxstyle='round', facecolor='#f4f4f4', alpha=0.9, edgecolor='gray')
    ax.text(0.95, 0.95, summary_text, transform=ax.transAxes, fontsize=11,
            verticalalignment='top', horizontalalignment='right', bbox=props)


    plt.tight_layout()
    plt.show()

In [ ]:
plot_bleu_score(rezultat_bleu)


In [ ]:
def plot_rouge_score(rouge_dict):
    # 1. Extragem datele și le transformăm în procente
    r1 = round(rouge_dict['rouge1'] * 100, 2)
    r2 = round(rouge_dict['rouge2'] * 100, 2)
    rl = round(rouge_dict['rougeL'] * 100, 2)
    rlsum = round(rouge_dict['rougeLsum'] * 100, 2)

    scoruri = [r1, r2, rl, rlsum]

    # 2. Setăm dimensiunea graficului
    fig, ax = plt.subplots(figsize=(8, 5))

    # 3. Creăm barele pentru metricele ROUGE
    x_labels = ['ROUGE-1\n(Cuvinte)', 'ROUGE-2\n(Bigrame)', 'ROUGE-L\n(Cea mai lungă\nsecvență)', 'ROUGE-Lsum']
    culori = ['#4C72B0', '#55A868', '#C44E52', '#8172B2']
    bars = ax.bar(x_labels, scoruri, color=culori)

    # 4. Adăugăm procentele deasupra fiecărei bare
    for bar in bars:
        yval = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2, yval + 1.5, f'{yval}%',
                ha='center', va='bottom', fontweight='bold')

    # 5. Formatăm aspectul (axe, titlu, grid)
    ax.set_ylim(0, 100)
    ax.set_ylabel('Scor (%)', fontsize=12)
    ax.set_title('Analiza Scorurilor ROUGE', fontsize=14, fontweight='bold')
    ax.grid(axis='y', linestyle='--', alpha=0.7)

    # 6. Adăugăm o casetă cu un mini-rezumat / legendă
    summary_text = (f"Ghid de interpretare:\n"
                    f"ROUGE-1: Vocabular / Idei cheie\n"
                    f"ROUGE-2: Gramatică / Fluență\n"
                    f"ROUGE-L: Structura propoziției")

    props = dict(boxstyle='round', facecolor='#f4f4f4', alpha=0.9, edgecolor='gray')
    ax.text(0.95, 0.95, summary_text, transform=ax.transAxes, fontsize=11,
            verticalalignment='top', horizontalalignment='right', bbox=props)

    # 7. Afișăm graficul
    plt.tight_layout()
    plt.show()

# Apelarea corectă folosind variabila ta anterioară:
# plot_rouge_score(rezultate)

In [ ]:
plot_rouge_score(rezultat_rouge)

In [ ]:
def compute_metrics(eval_pred):
    predictions, labels = eval_pred

    # Decodăm predicțiile modelului (rezumatele generate)
    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)

    # Înlocuim -100 din etichete (labels) cu token-ul de padding pentru a le putea decoda
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    # Curățăm marginile de spații inutile
    decoded_preds = [pred.strip() for pred in decoded_preds]
    decoded_labels = [label.strip() for label in decoded_labels]

    # 2. Calculăm TER (Translation Edit Rate)
    result_ter = ter_metric.compute(predictions=decoded_preds, references=decoded_labels)

    # 3. Returnăm direct scorul TER, rotunjit la 4 zecimale pentru un tabel curat
    return {"ter": round(result_ter["score"], 4)}